# Multicellular simulations, one W at a time (OG, local)

Readable version of the multicellular model. Runs every GRN at every γ for **one** intercellular
kernel `W` and saves one `.jld2` file per variable.

Runs on a laptop, no server needed. For the full W bank (1000 Ws) use the optimised pipeline in
`Model/opti/run_sims/` instead: the same model, but this version would take far too long for that many Ws.

`run_nsims_final(σ_init, W, γ; K_gene, β, n_sims, n_sweeps, ...)` runs `n_sims` simulations of the selfish
dynamics from the same start `σ_init` and stops each one at `n_sweeps` or once it is frozen / quasi-stable.
It returns `σ_finals_selfish`, `E_selfish`, `avg_flip_counts`, `avg_site_flip_counts`,
`avg_tissue_frustration`, `sweeps_used`, `equilibrated` and `n_active`.


In [ ]:
include("../functions/functions_frustration_index_OG.jl")              # frustration index (used every sweep)
include("../functions/functions_ising_model_finalsweeps_gamma_flipequ_OG.jl")  # dynamics

using JLD2

base_path = "results_OG"     # output folder, relative to this notebook
mkpath(base_path)

function make_filename(base_path, varname, W, K, β, γ, σ, n_sims, max_sweeps)
    return joinpath(base_path,
        "$(varname)_$(W)_$(K)_$(β)_$(γ)_$(σ)_$(n_sims)_max$(max_sweeps)sweeps_flipequ.jld2")
end


## The intercellular kernel W
One W from the W bank used in the paper (`W_bank.jld2`, made by `Model/opti/run_sims/make_W_bank_opti.jl`),
or a new random one.


In [ ]:
W_bank_file = "../../../data/W_bank.jld2"
W_idx       = 1

W_test = load(W_bank_file, "W")[:, :, W_idx]
W_name = "W$(W_idx)"

# or a new random symmetric W (uniform on [-1, 1]):
# W_test = compute_W(9)
# W_name = "Wrand"


## GRNs
Rows are inputs: `K[A, B]` is the effect of gene `B` on gene `A`. Genes 4–9 are the same in every GRN;
the GRNs differ only in the 1–2–3 module.


In [ ]:
K_tree = Float64.([
    0  1  1  0  0  0  0  0  0;
    0  0  0  1  0  0  0  0  0;
    0  0  0  0  1  1  0  0  0;
    0  0  0  0  0  0 -1  0  0;
    0  0  0  0  0  0  0 -1  0;
    0  0  0  0  0  0  0  0 -1;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0;
    0  0  0  0  0  0  0  0  0
])

K_negtri = Float64.([
    0   1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_ICFF = Float64.([
    0   1  -1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0   0  0  0  0 -1  0  0;
    0   0   0  0  0  0  0 -1  0;
    0   0   0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_CFF = Float64.([
    0   1   1  0  0  0  0  0  0;
    0   0   1  1  0  0  0  0  0;
    0   0   0  0  0  1  0  0  0;
    0   0   0  0  0  0 -1  0  0;
    0   0   0  0  0  0  0 -1  0;
    0   0   0  0  0  0  0  0 -1;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0;
    0   0   0  0  0  0  0  0  0
])

K_negtritri = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  -1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

K_negtritwin = Float64.([
    0   -1  0  0  0  0  0  0  0;
    0   0  1  1  0  0  0  0  0;
   -1   0  0  0  0  1  0  0  0;
    0   0  0  0  0  0 -1  0  0;
    0   0  0  0  0  0  0 -1  0;
    0   0  0  0  0  0  0  0 -1;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0;
    0   0  0  0  0  0  0  0  0
])

GRNs = [
    ("tree",       K_tree),
    ("negtri",     K_negtri),
    ("ICFF",       K_ICFF),
    ("CFF",        K_CFF),
    ("negtritwin", K_negtritwin),
    ("negtritri",  K_negtritri),
]


## Settings


In [ ]:
### ----------- lattice and runs ----------- ###
n_cells = 400        # 20×20 lattice
n_genes = 9
n_sims  = 10
max_sweeps = 10_000

β_test = Inf
β_name = "betaInf"

# stopping rule (same values as the optimised pipeline)
stop_kwargs = (equilibrium_window = 30, flip_tol = 5, oscillation_window = 100, flip_range_tol = 5)

### ----------- initial state ----------- ###
σ_test = rand([-1, 1], n_cells, n_genes)    # each spin ±1 with probability 1/2
σ_name = "σrand"

### ----------- γ values ----------- ###
gammas      = [1.0, 0.8, 0.5, 0.3, 0.25, 0.22, 0.2, 0.18, 0.16, 0.14, 0.125, 0.12, 0.1, 0.08, 0.07, 0.06, 0.05, 0.04, 0.001, 0.0]
gamma_names = ["g1", "g08", "g05", "g03", "g025", "g022", "g02", "g018", "g016", "g014", "g0125", "g012", "g01", "g008", "g007", "g006", "g005", "g004", "g0001", "g0"]


## Run
Every GRN at every γ. Each (γ, GRN) takes a while with this readable code; shorten `gammas` / `GRNs`
above to run a subset.


In [ ]:
for (γ, γ_name) in zip(gammas, gamma_names), (K_name, K_test) in GRNs
    println("γ = $(γ), $(K_name)")

    @time results = run_nsims_final(
        copy(σ_test),
        W_test,
        γ;
        K_gene   = K_test,
        β        = β_test,
        n_sims   = n_sims,
        n_sweeps = max_sweeps,
        stop_kwargs...
    )

    vars = Dict(
        "sigma_finals_selfish"   => results.σ_finals_selfish,
        "E_selfish"              => results.E_selfish,
        "avg_flip_counts"        => results.avg_flip_counts,
        "avg_site_flip_counts"   => results.avg_site_flip_counts,
        "avg_tissue_frustration" => results.avg_tissue_frustration,
        "sweeps_used"            => results.sweeps_used,
        "equilibrated"           => results.equilibrated,
        "n_active"               => results.n_active
    )

    for (name, val) in vars
        filename = make_filename(base_path, name, W_name, K_name, β_name, γ_name, σ_name, n_sims, max_sweeps)
        @save filename val
    end
end
